# TheMergeConflict Final Solution Notebook

Team: **TheMergeConflict**  
Members: **Jnanasree Konda (NetID: jk9286)** and **Rithwik Amajala (NetID: sa9880)**  
Public leaderboard position: **6**  
Public score: **0.915**  
Final submitted CSV: `submission.csv`

This notebook is the single final-code reproducibility artifact. It uses only the provided competition data and the allowed model `HuggingFaceTB/SmolVLM-500M-Instruct`.

Run the notebook from top to bottom in a fresh GPU runtime. The final pipeline trains a LoRA adapter, selects the best validation checkpoint, scores each answer option with constrained answer logits, applies choice-permutation test-time augmentation, averages compatible score matrices from the final run, writes `submission.csv`, and validates the file format.

In [ ]:
# If running in a fresh Kaggle/Colab runtime, run this cell once and restart the runtime if packages change.
# In the offline evaluation environment, these packages/model files should already be attached locally.
# !pip install -q transformers==4.57.6 peft==0.18.1 accelerate datasets pandas numpy tqdm Pillow safetensors

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from dataclasses import dataclass
from pathlib import Path
import csv
import math
import random
from typing import Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoProcessor, AutoModelForVision2Seq, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

In [ ]:
@dataclass
class CFG:
    # Central experiment configuration. Change only data_dir/output_dir unless you
    # intentionally want to rerun a different controlled experiment.
    # data_dir must point to the provided competition package; no external data is used.
    data_dir: Path = Path("/kaggle/input/pixels-to-predictions")
    # All adapters, score matrices, and submission.csv are written here.
    output_dir: Path = Path("./outputs")
    model_id: str = "HuggingFaceTB/SmolVLM-500M-Instruct"
    seed: int = 7
    # 384 was the stable image long-side setting: high enough for diagrams,
    # small enough to avoid Idefics3 processor/CUDA failures on free GPUs.
    image_size: int = 384
    max_seq_len: int = 1024
    epochs: int = 3
    micro_batch_size: int = 1
    grad_accum: int = 16
    lr: float = 1e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05
    # LoRA parameters stay well below the 5M trainable-parameter competition cap.
    lora_rank: int = 8
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj", "k_proj", "v_proj", "o_proj")
    max_trainable_params: int = 5_000_000
    eval_batch_size: int = 1
    tta_permutations: int = 8

cfg = CFG()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
def find_file(root: Path, names):
    for name in names:
        p = root / name
        if p.exists():
            return p
    matches = []
    for name in names:
        matches.extend(root.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find any of {names} under {root}")
    return matches[0]

train_csv = find_file(cfg.data_dir, ["train.csv"])
val_csv = find_file(cfg.data_dir, ["validation.csv", "val.csv"])
test_csv = find_file(cfg.data_dir, ["test.csv"])

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)
print(train_csv, val_csv, test_csv)
print("rows:", len(train_df), len(val_df), len(test_df))
print("train columns:", list(train_df.columns))

In [ ]:
# Data parsing utilities. The competition files may use slightly different
# column names, so these helpers choose the first available field without
# changing the actual provided examples.
def first_present(row, candidates, default=""):
    for c in candidates:
        if c in row and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c]).strip()
    return default

def parse_choices(row):
    # Handles either a list-like choice column or separate option columns.
    if "choices" in row and pd.notna(row["choices"]):
        val = row["choices"]
        if isinstance(val, str):
            try:
                import ast
                parsed = ast.literal_eval(val)
                if isinstance(parsed, (list, tuple)):
                    return [str(x) for x in parsed]
            except Exception:
                pass
    choices = []
    for c in ["A", "B", "C", "D", "E", "choice_A", "choice_B", "choice_C", "choice_D", "choice_E", "option_0", "option_1", "option_2", "option_3", "option_4"]:
        if c in row and pd.notna(row[c]) and str(row[c]).strip():
            choices.append(str(row[c]).strip())
    if not choices:
        for c in row.index:
            if str(c).lower().startswith(("choice", "option")) and pd.notna(row[c]):
                choices.append(str(row[c]).strip())
    return choices

def image_path_for_row(row):
    image_name = first_present(row, ["image", "image_path", "image_file", "filename", "file_name"])
    if not image_name:
        return None
    p = Path(image_name)
    if p.exists():
        return p
    for folder in [cfg.data_dir, cfg.data_dir / "images", cfg.data_dir / "train", cfg.data_dir / "test"]:
        candidate = folder / image_name
        if candidate.exists():
            return candidate
    matches = list(cfg.data_dir.rglob(Path(image_name).name))
    return matches[0] if matches else None

def load_image(row):
    p = image_path_for_row(row)
    if p is None:
        return Image.new("RGB", (cfg.image_size, cfg.image_size), color="white")
    im = Image.open(p).convert("RGB")
    w, h = im.size
    scale = cfg.image_size / max(w, h)
    if scale != 1:
        im = im.resize((max(1, int(w * scale)), max(1, int(h * scale))))
    return im

LETTERS = list("ABCDE")

def build_prompt(row, choices, include_solution=False):
    q = first_present(row, ["question", "Question", "prompt"])
    hint = first_present(row, ["hint", "Hint"])
    lecture = first_present(row, ["lecture", "Lecture", "context"])
    solution = first_present(row, ["solution", "Solution", "explanation"])
    parts = ["Select the correct answer. Return only the answer letter.", f"Question: {q}"]
    if hint:
        parts.append(f"Hint: {hint}")
    if lecture:
        parts.append(f"Lecture: {lecture}")
    parts.append("Choices:")
    for i, choice in enumerate(choices):
        parts.append(f"{LETTERS[i]}. {choice}")
    if include_solution and solution:
        parts.append(f"Solution context: {solution[:500]}")
    parts.append("Answer:")
    return "\n".join(parts)

In [ ]:
# Processor and model setup. The processor settings below intentionally avoid
# image-splitting/resize edge cases that caused runtime-dependent failures while
# preserving the same visual information scale used in the high-scoring run.
processor = AutoProcessor.from_pretrained(cfg.model_id)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
if hasattr(processor, "image_processor") and hasattr(processor.image_processor, "do_image_splitting"):
    processor.image_processor.do_image_splitting = False
if hasattr(processor, "image_processor"):
    # Avoid Idefics3 resize mismatch errors across transformer versions.
    processor.image_processor.size = {"longest_edge": cfg.image_size}
    processor.image_processor.max_image_size = {"longest_edge": cfg.image_size}

base_model = AutoModelForVision2Seq.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
if not torch.cuda.is_available():
    base_model.to(device)
base_model.config.use_cache = False
for p in base_model.parameters():
    p.requires_grad = False

lora_cfg = LoraConfig(
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=list(cfg.lora_targets),
)
model = get_peft_model(base_model, lora_cfg)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params:", trainable)
assert trainable <= cfg.max_trainable_params
model.print_trainable_parameters()

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        choices = parse_choices(row)
        answer = int(row["answer"]) if self.is_train and "answer" in row else -1
        return {"row": row, "choices": choices, "answer": answer, "image": load_image(row), "id": first_present(row, ["id"])}

train_ds = MCQDataset(train_df, is_train=True)
val_ds = MCQDataset(val_df, is_train=True)
test_ds = MCQDataset(test_df, is_train=False)

def make_train_batch(items):
    texts, images = [], []
    for item in items:
        ans_letter = LETTERS[item["answer"]]
        prompt = build_prompt(item["row"], item["choices"])
        full = prompt + " " + ans_letter
        texts.append(full)
        images.append([item["image"]])
    enc = processor(text=texts, images=images, return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_seq_len)
    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    return enc

train_loader = DataLoader(train_ds, batch_size=cfg.micro_batch_size, shuffle=True, collate_fn=make_train_batch, num_workers=0)
print("steps/epoch:", len(train_loader))

In [ ]:
# Final training loop: train the LoRA adapter and keep the checkpoint with best validation accuracy.
def letter_token_id(letter):
    # We score the logits of answer letters directly instead of generating free-form text.
    # This avoids parsing errors and exactly matches the multiple-choice objective.
    ids = processor.tokenizer(letter, add_special_tokens=False).input_ids
    if len(ids) != 1:
        ids = processor.tokenizer(" " + letter, add_special_tokens=False).input_ids
    return ids[-1]
LETTER_IDS = {L: letter_token_id(L) for L in LETTERS}

@torch.inference_mode()
def score_items(model, items):
    model.eval()
    processor.tokenizer.padding_side = "left"
    texts, images, counts = [], [], []
    for item in items:
        texts.append(build_prompt(item["row"], item["choices"]))
        images.append([item["image"]])
        counts.append(len(item["choices"]))
    enc = processor(text=texts, images=images, return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_seq_len)
    enc = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in enc.items()}
    out = model(**enc)
    last = out.logits[:, -1, :]
    scores = np.full((len(items), max(counts)), -1e9, dtype=np.float32)
    for i, n in enumerate(counts):
        scores[i, :n] = last[i, [LETTER_IDS[LETTERS[j]] for j in range(n)]].float().cpu().numpy()
    return scores

@torch.inference_mode()
def evaluate(model, ds, max_samples=None):
    n = len(ds) if max_samples is None else min(max_samples, len(ds))
    correct = total = 0
    for start in range(0, n, cfg.eval_batch_size):
        items = [ds[i] for i in range(start, min(n, start + cfg.eval_batch_size))]
        labels = np.array([it["answer"] for it in items])
        preds = score_items(model, items).argmax(axis=1)
        correct += int((preds == labels).sum())
        total += len(items)
    return correct / max(total, 1)

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
total_updates = math.ceil(len(train_loader) / cfg.grad_accum) * cfg.epochs
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_updates * cfg.warmup_ratio), total_updates)

best_acc = -1
best_dir = cfg.output_dir / "best_adapter"
optimizer.zero_grad(set_to_none=True)
step_count = 0
for epoch in range(cfg.epochs):
    model.train()
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{cfg.epochs}")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / cfg.grad_accum
        loss.backward()
        if step % cfg.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
            step_count += 1
            pbar.set_postfix(loss=float(loss.detach().cpu()) * cfg.grad_accum)
    acc = evaluate(model, val_ds)
    print(f"epoch {epoch+1}: val_acc={acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
print("best validation accuracy:", best_acc, "adapter:", best_dir)

In [ ]:
# Final inference: choice-permutation test-time augmentation with score remapping.
def permuted_items(item, n_perm):
    # SmolVLM can have mild position bias toward earlier/later options. For each
    # test row we score several deterministic choice orders, then map scores
    # back to the original option indices before averaging.
    choices = item["choices"]
    base = list(range(len(choices)))
    yield choices, base
    rng = np.random.RandomState(cfg.seed + abs(hash(item["id"])) % 100000)
    seen = {tuple(base)}
    while len(seen) < min(n_perm, math.factorial(len(choices))):
        perm = base[:]
        rng.shuffle(perm)
        t = tuple(perm)
        if t in seen:
            continue
        seen.add(t)
        yield [choices[i] for i in perm], perm

@torch.inference_mode()
def score_tta(model, ds):
    all_scores = []
    ids = []
    for idx in tqdm(range(len(ds)), desc="TTA inference"):
        item = ds[idx]
        n = len(item["choices"])
        accum = np.zeros(n, dtype=np.float32)
        used = 0
        for choices_perm, perm in permuted_items(item, cfg.tta_permutations):
            item2 = dict(item)
            item2["choices"] = choices_perm
            s = score_items(model, [item2])[0]
            back = np.zeros(n, dtype=np.float32)
            for new_pos, old_pos in enumerate(perm):
                back[old_pos] = s[new_pos]
            accum += back
            used += 1
        all_scores.append(accum / max(used, 1))
        ids.append(item["id"])
    max_choices = max(len(s) for s in all_scores)
    mat = np.full((len(all_scores), max_choices), -1e9, dtype=np.float32)
    for i, s in enumerate(all_scores):
        mat[i, :len(s)] = s
    return ids, mat

base = AutoModelForVision2Seq.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
infer_model = PeftModel.from_pretrained(base, cfg.output_dir / "best_adapter")
infer_model.eval()
ids, scores = score_tta(infer_model, test_ds)
np.save(cfg.output_dir / "answer_scores.npy", scores)
print(scores.shape)

In [ ]:
# Final score averaging and submission export.
# The final notebook writes one answer-score matrix. If the runtime is used to
# reproduce additional allowed seed/checkpoint matrices, place them in this list
# before running the cell; all matrices must have the same test-row order/shape.
score_paths = [cfg.output_dir / "answer_scores.npy"]

score_arrays = [np.load(p) for p in score_paths]
assert all(a.shape == score_arrays[0].shape for a in score_arrays)
blended = sum(score_arrays) / len(score_arrays)
preds = blended.argmax(axis=1).astype(int)

submission_path = cfg.output_dir / "submission.csv"
with submission_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    writer.writerows(zip(ids, preds))
print("wrote", submission_path)

In [ ]:
# Final validation cell: required competition format.
def check_submission(path, expected_rows=None):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    assert reader.fieldnames == ["id", "answer"], reader.fieldnames
    if expected_rows is not None:
        assert len(rows) == expected_rows, len(rows)
    assert len({r["id"] for r in rows}) == len(rows)
    answers = [int(r["answer"]) for r in rows]
    assert min(answers) >= 0
    from collections import Counter
    print("OK", path)
    print("rows", len(rows))
    print("answer distribution", dict(sorted(Counter(answers).items())))

check_submission(cfg.output_dir / "submission.csv", expected_rows=len(test_df))